# Train the MLBB vision detector (free, in Google Colab)

Makes `mlbb_detector.tflite` + `labels.txt` for the MLBB Research app (Model page → *Choose detector file*).

**Before you start**
1. *Runtime → Change runtime type → T4 GPU* (free).
2. Make a free account at roboflow.com and copy your **API key** (Settings → API Keys). You type it into the box below; it is never saved in this notebook.
3. Default dataset: **Mobile Legends Minimap Dataset** by *gladi* on Roboflow Universe, licence **CC BY 4.0** (credit the author if you share the model).

Run each cell from top to bottom (tap ▶). Training takes about 10–20 minutes.

In [ ]:
!pip install -q ultralytics roboflow

## 1. Settings
Change these only if you want a different Roboflow dataset. For a full-screen dataset, also set the app's *Detector looks at* to *the whole screen*.

In [ ]:
WORKSPACE = "gladi"                            # Roboflow workspace (from the dataset URL)
PROJECT   = "mobile-legends-minimap-dataset"   # Roboflow project (from the dataset URL)
VERSION   = 2                                  # dataset version number
EPOCHS    = 80                                 # more = better but slower
IMGSZ     = 320                                # 320 is fast on phones; 640 is more accurate

## 2. Download the dataset

In [ ]:
from getpass import getpass
from roboflow import Roboflow

rf = Roboflow(api_key=getpass("Paste your Roboflow API key and press Enter: "))
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")
print("Downloaded to", dataset.location)

## 3. Train a small YOLOv8 model

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")   # 'nano' = smallest and fastest, good for phones
model.train(data=f"{dataset.location}/data.yaml", epochs=EPOCHS, imgsz=IMGSZ, batch=16, patience=20)
best_path = model.trainer.best
print("Best weights:", best_path)

## 4. Export to LiteRT (.tflite) and download

In [ ]:
import glob, shutil, yaml
from google.colab import files

best = YOLO(best_path)
best.export(format="tflite", imgsz=IMGSZ)          # float32 input/output, works on any phone

tflite = sorted(glob.glob(str(best_path).replace("best.pt", "best_saved_model/*float32.tflite")))[0]
shutil.copy(tflite, "mlbb_detector.tflite")

names = yaml.safe_load(open(f"{dataset.location}/data.yaml"))["names"]
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]
open("labels.txt", "w").write(",".join(names))
print("Classes (paste into the app if they differ from the default):")
print(",".join(names))

files.download("mlbb_detector.tflite")
files.download("labels.txt")

## 5. Put it in the app
1. Copy `mlbb_detector.tflite` to your phone (it downloads straight to the phone if you run Colab there).
2. App → **Model** page → *Choose detector file (.tflite)*.
3. If the class list printed above differs from the app's default, paste it into *Class names*.
4. Home page preview shows the detections as cyan boxes.